# Solar Energy Self-Consumption Analytics
## STUDENT B — METRICS
Name: Akash Kumar
Role: Calculate energy metrics and daily/building summaries.

 Rules for my role
- Work on a copy of Student A's cleaned DataFrame; do not change the input.
- Calculate self-consumed energy as generation minus export.
- Calculate balance gap as generation plus import minus load minus export.
- If generation is zero, show the percentage as NaN because it is undefined.
- For grouped percentages, sum the numerator and denominator first; do not average row percentages.
- Keep required column names and full precision; sort daily results by date and entities by ID.
- Retain the documented zero-activity row and do not assume cross-day balance continuity.


Responsibilities

- Calculate self-consumed energy and the balance gap for each building-day.
- Calculate self-consumption percentage safely; a zero generation denominator means the rate is undefined (`NaN`).
- Summarize source quantities by date and building.
- Calculate grouped percentages from summed numerators and denominators, not by averaging percentages.
- Preserve the cleaned input and use the exact output columns in `12_dictionary.txt`.


In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / '02_data.csv').exists(), 'Open this notebook from the NGF-Project-04-main folder.'

def load(filename):
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

RUN_MY_CHECKS = True
checks = load('11_checks.py')


## Implementation decisions

1. I return `NaN` when generation is zero because a self-consumption percentage is undefined in that case; calling it 0% would give it a false meaning.
2. I sum the rate numerator and denominator before dividing for daily and entity summaries. This weights the result by generated energy.


In [ ]:
# Inspect the actual project data and the cleaning output supplied by Student A.
raw = pd.read_csv(ROOT / '02_data.csv', dtype=str, keep_default_na=False)
print('Raw rows:', len(raw))
print(raw.head().to_string(index=False))
a = load('07_student_A.py')
clean, audit = a.load_and_clean(ROOT / '02_data.csv')
print('Clean rows:', len(clean))
print('Audit actions:')
print(audit['action'].value_counts())


## Bug fixed

The original `08_student_B.py` contained a placeholder that raised `NotImplementedError`. I replaced it with the metric calculations and safe division helper, then use the project checks below to check the output.


In [ ]:
# A hand-worked row: generation=100 kWh, export=20 kWh.
# Self-consumed energy = 100 - 20 = 80 kWh; rate = 80 / 100 * 100 = 80%.
example = pd.DataFrame({
    'date': pd.to_datetime(['2026-07-01']),
    'entity_id': ['E01'],
    'generation_kwh': [100.0],
    'load_kwh': [100.0],
    'export_kwh': [20.0],
    'import_kwh': [20.0],
})
b = load('08_student_B.py')
example_detail, example_daily, example_entities = b.calculate_metrics(example)
assert example_detail.loc[0, 'self_consumed_kwh'] == 80.0
assert example_detail.loc[0, 'self_consumption_pct'] == 80.0
assert example_detail.loc[0, 'balance_gap_kwh'] == 0.0
print(example_detail[['self_consumed_kwh', 'balance_gap_kwh', 'self_consumption_pct']])

## Added edge-case checks

This fixture uses unequal generation amounts so a simple average would give the wrong grouped rate. It also includes a zero-generation day, whose rate should be `NaN`.


In [ ]:
fixture = pd.DataFrame({
    'date': pd.to_datetime(['2026-07-01', '2026-07-01', '2026-07-02']),
    'entity_id': ['E01', 'E02', 'E03'],
    'generation_kwh': [100.0, 10.0, 0.0],
    'load_kwh': [80.0, 10.0, 0.0],
    'export_kwh': [20.0, 0.0, 0.0],
    'import_kwh': [0.0, 0.0, 0.0],
})
fd, fday, fent = b.calculate_metrics(fixture)
# Weighted July 1 rate = ((100-20) + (10-0)) / (100+10) * 100 = 81.818...%
assert np.isclose(fday.loc[0, 'weighted_self_consumption_pct'], 100 * 90 / 110)
assert np.isnan(fd.loc[2, 'self_consumption_pct'])
assert np.isnan(fday.loc[1, 'weighted_self_consumption_pct'])
print(fday[['date', 'weighted_self_consumption_pct']])


In [ ]:
if RUN_MY_CHECKS:
    checks.mini_b(b.calculate_metrics, clean)
    detail, daily, entities = b.calculate_metrics(clean)
    checks.check_b(detail, daily, entities)
    print('Student B checks passed.')
    print('Daily rows:', len(daily), '| Entity rows:', len(entities))
    print(entities[['entity_id', 'weighted_self_consumption_pct']].to_string(index=False))

## Explain one result

A higher self-consumption rate means a greater share of generated solar energy was used locally in the daily meter totals. This rate alone does not establish tariff savings or battery needs: daily netting hides within-day imports and exports, and these are synthetic teaching data.

**Submission:** Save this executed notebook and `08_student_B.py`, and provide the detail, daily, and entity outputs with the group project.
